In [1]:
!pip install torch torchvision transformers datasets -q
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
print(torch.__version__, "GPU:", torch.cuda.is_available())

2.11.0+cu128 GPU: True


In [2]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, h=8):
        super().__init__()
        self.h = h
        self.d_k = d_model // h
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.h, self.d_k)
        return x.transpose(1, 2)  # (batch, h, seq_len, d_k)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)
        Q = self.split_heads(self.W_q(Q), batch_size)
        K = self.split_heads(self.W_k(K), batch_size)
        V = self.split_heads(self.W_v(V), batch_size)
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.h * self.d_k)
        return self.W_o(attn_output), attn_weights

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model=512, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [5]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model=512, h=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, h)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, _ = self.attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

In [6]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_model=512, h=8, d_ff=2048, N=6):
        super().__init__()
        self.pos_enc = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, h, d_ff) for _ in range(N)])

    def forward(self, x, mask=None):
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, mask)
        return x

# Quick sanity test
encoder = TransformerEncoder()
dummy_input = torch.randn(2, 10, 512)   # (batch=2, seq_len=10, d_model=512)
out = encoder(dummy_input)
print(out.shape)  # should be torch.Size([2, 10, 512])

torch.Size([2, 10, 512])


In [7]:
def generate_causal_mask(seq_len):
    # Lower-triangular matrix: 1 = allowed to attend, 0 = blocked
    mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
    return mask  # shape (1, 1, seq_len, seq_len), broadcasts over batch and heads

In [8]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model=512, h=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, h)   # masked self-attention
        self.cross_attn = MultiHeadAttention(d_model, h)  # encoder-decoder attention
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, tgt_mask=None, src_mask=None):
        # 1) Masked self-attention: Q,K,V all from the decoder's own sequence so far
        attn_out, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_out))

        # 2) Cross-attention: Q from decoder, K & V from ENCODER output (Section 3.2.3 bullet 1)
        cross_out, cross_weights = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_out))

        # 3) Feed-forward
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_out))
        return x, cross_weights

In [9]:
class TransformerDecoder(nn.Module):
    def __init__(self, d_model=512, h=8, d_ff=2048, N=6):
        super().__init__()
        self.pos_enc = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([DecoderLayer(d_model, h, d_ff) for _ in range(N)])

    def forward(self, x, encoder_output, tgt_mask=None, src_mask=None):
        x = self.pos_enc(x)
        attn_weights = None
        for layer in self.layers:
            x, attn_weights = layer(x, encoder_output, tgt_mask, src_mask)
        return x, attn_weights

In [10]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, h=8, d_ff=2048, N=6):
        super().__init__()
        self.d_model = d_model
        self.src_embed = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model)
        self.encoder = TransformerEncoder(d_model, h, d_ff, N)
        self.decoder = TransformerDecoder(d_model, h, d_ff, N)
        self.output_linear = nn.Linear(d_model, tgt_vocab_size)  # "Linear" box in Figure 1

    def forward(self, src, tgt):
        tgt_len = tgt.size(1)
        tgt_mask = generate_causal_mask(tgt_len)

        # Embeddings are scaled by sqrt(d_model), as stated in Section 3.4
        src_emb = self.src_embed(src) * (self.d_model ** 0.5)
        tgt_emb = self.tgt_embed(tgt) * (self.d_model ** 0.5)

        encoder_output = self.encoder(src_emb)
        decoder_output, attn_weights = self.decoder(tgt_emb, encoder_output, tgt_mask)

        logits = self.output_linear(decoder_output)
        return logits  # apply softmax outside (e.g. F.softmax(logits, dim=-1)) — "Softmax" box in Figure 1

In [11]:
src_vocab_size = 1000
tgt_vocab_size = 1000

model = Transformer(src_vocab_size, tgt_vocab_size)

src = torch.randint(0, src_vocab_size, (2, 10))  # (batch=2, src_seq_len=10) — token IDs
tgt = torch.randint(0, tgt_vocab_size, (2, 8))   # (batch=2, tgt_seq_len=8)

logits = model(src, tgt)
print(logits.shape)  # should be torch.Size([2, 8, 1000]) -> (batch, tgt_seq_len, vocab_size)

torch.Size([2, 8, 1000])
